Import Library

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split,cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV

Import Data

In [5]:
data = pd.read_csv('encoded_data.csv')
print(data.info())
data.head()
data_model = data[data['Cluster_ID_Outlier'] == 0]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Columns: 139 entries, Profession_ข้าราชการ to T_Trial_ไม่ลอง
dtypes: float64(82), int64(57)
memory usage: 181.5 KB
None


predict customer segment

In [6]:
labels = data_model[['Cluster_ID_ชอบกาแฟ', 'Cluster_ID_ชอบชา', 'Cluster_ID_ชอบทั้งกาแฟและชา']].idxmax(axis=1)


In [7]:
print(labels)

1      Cluster_ID_ชอบทั้งกาแฟและชา
2               Cluster_ID_ชอบกาแฟ
3      Cluster_ID_ชอบทั้งกาแฟและชา
4      Cluster_ID_ชอบทั้งกาแฟและชา
5                 Cluster_ID_ชอบชา
                  ...             
161               Cluster_ID_ชอบชา
162    Cluster_ID_ชอบทั้งกาแฟและชา
163    Cluster_ID_ชอบทั้งกาแฟและชา
164               Cluster_ID_ชอบชา
166    Cluster_ID_ชอบทั้งกาแฟและชา
Length: 163, dtype: object


In [8]:
age_cols = ['Age']
sex_cols = [c for c in data_model.columns if c.startswith('Sex_')]
prof_cols = [c for c in data_model.columns if c.startswith('Profession_')]
prov_cols = [c for c in data_model.columns if c.startswith('Province_')]

feature_cols = age_cols + sex_cols + prof_cols + prov_cols

X = data_model[feature_cols]

y = labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [9]:
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [100, 200, None],
    'class_weight': ['balanced']
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=10, scoring='accuracy')
grid_search.fit(X_train, y_train)

best_rf_new = grid_search.best_estimator_
y_pred_new = best_rf_new.predict(X_test)

print("--- Customer segment  ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_new)}")
print(classification_report(y_test, y_pred_new))

--- Customer segment  ---
Accuracy : 0.42857142857142855
                             precision    recall  f1-score   support

         Cluster_ID_ชอบกาแฟ       0.13      0.33      0.19         6
           Cluster_ID_ชอบชา       0.47      0.40      0.43        20
Cluster_ID_ชอบทั้งกาแฟและชา       0.65      0.48      0.55        23

                   accuracy                           0.43        49
                  macro avg       0.42      0.40      0.39        49
               weighted avg       0.51      0.43      0.46        49



predict platform

In [10]:
platform_cols = [c for c in data_model.columns if c.startswith('S_Occasion_')]

X = data_model[feature_cols]
y = data_model[platform_cols].idxmax(axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
#random forest

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("--- Platform  ---")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

--- Platform  ---
Accuracy: 0.3673469387755102
                      precision    recall  f1-score   support

 S_Occasion_Facebook       0.53      0.60      0.56        15
S_Occasion_Instagram       0.43      0.27      0.33        11
     S_Occasion_LINE       0.22      0.42      0.29        12
   S_Occasion_TikTok       0.50      0.25      0.33         4
  S_Occasion_Twitter       0.00      0.00      0.00         7

            accuracy                           0.37        49
           macro avg       0.34      0.31      0.30        49
        weighted avg       0.35      0.37      0.34        49



C:\Users\MSI\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\MSI\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\MSI\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:

predict time to ads timing

In [12]:
time_cols = [c for c in data_model.columns if c.startswith('S_Time(weekday)_')]

y = data_model[time_cols].idxmax(axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [13]:
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("--- Ad Timing ---")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

--- Ad Timing ---
Accuracy: 0.3469387755102041
                               precision    recall  f1-score   support

 S_Time(weekday)_05.00-8.59น.       0.17      0.10      0.12        10
S_Time(weekday)_18.00-21.59น.       0.54      0.50      0.52        30
 S_Time(weekday)_22.01-4.59น.       0.00      0.00      0.00         7
 S_Time(weekday)_9.00-17.59น.       0.07      0.50      0.12         2

                     accuracy                           0.35        49
                    macro avg       0.19      0.28      0.19        49
                 weighted avg       0.36      0.35      0.35        49



save model

In [14]:
import joblib

# save ทั้ง 3 model
joblib.dump(best_rf_new, "customer_model.pkl")
joblib.dump(rf_model, "platform_model.pkl")
joblib.dump(model, "time_model.pkl")

# save feature columns ด้วย (สำคัญมาก)
joblib.dump(feature_cols, "feature_cols.pkl")

['feature_cols.pkl']

In [15]:
customer_labels = ['ชอบกาแฟ', 'ชอบชา', 'ชอบทั้งกาแฟและชา']

platform_labels = [c.replace('S_Occasion_', '') for c in platform_cols]

time_labels = [c.replace('S_Time(weekday)_', '') for c in time_cols]

import joblib
joblib.dump(customer_labels, "customer_labels.pkl")
joblib.dump(platform_labels, "platform_labels.pkl")
joblib.dump(time_labels, "time_labels.pkl")

['time_labels.pkl']